# 01 — Literature & Dataset Inventory

**AI-Enabled Concrete Performance Passports for OpenBIM**  
*Machine Learning–Based Quality, Durability, and Sustainability Data Management for Concrete Structures*

This notebook documents the **literature positioning** and the **public datasets** used by the project,
and runs lightweight **availability/loading checks**. It does not train models (see notebook 02) or
build the OpenBIM mapping (see notebook 03).

All sources and dataset properties are recorded in `outputs/literature_matrix.csv`,
`outputs/literature_summary.md`, `outputs/dataset_inventory.md`, and `outputs/dataset_inventory.csv`.

## 0. Google Colab setup

Run this first in Colab. It clones the repository (if needed) and installs light dependencies.
Locally, you can skip the clone and just run from the repo root.

In [ ]:
import os, sys, subprocess

# Detect Colab and clone the repo if the project files are not already present.
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/sayyedalimrj/Paper.git'  # update if forked
if IN_COLAB and not os.path.exists('Paper'):
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
    except Exception as e:
        print('Clone failed (you can upload the repo manually):', e)

# Move into the repo root if present.
for cand in ('Paper', '.'):
    if os.path.exists(os.path.join(cand, 'src', 'config.py')):
        os.chdir(cand)
        break
print('Working dir:', os.getcwd())

# Light install (most are pre-installed in Colab).
%pip install -q pandas numpy requests openpyxl xlrd 2>/dev/null || True

## 1. Literature positioning (summary)

Full matrix: `outputs/literature_matrix.csv` (35 sources, areas A–F). Key takeaways:

| Stream | Representative work | Where it usually stops |
|---|---|---|
| A. ML for concrete | Yeh 1998; ensemble/SHAP studies; RILEM TC 315-DCS durability review | At a metric/plot for one property; output not element-bound or interoperable |
| B. BIM/IFC QA/QC | IfcMaterial & concrete Psets; QualiSite (ISO 9001) | At inspection/document management, not predictive performance |
| C. Material passports | Honic et al. 2019; circular-economy MP reviews | At reuse / end-of-life inventory |
| D. BIM-LCA/EPD | BIM-LCA SLRs; EPD-in-IFC limitations; large EPD dataset | At carbon accounting; EPD weakly represented in IFC |
| E. Construction DT QA | construction-phase DT framework; maturity (ASTM C1074) | Element-aware but not via OpenBIM IFC/IDS |
| F. IDS | buildingSMART IDS v1.0; IDM (ISO 29481) | Generic mechanism awaiting domain requirements |

**Honest novelty:** transforming standard concrete ML predictions into an **OpenBIM-ready Concrete
Performance Passport** that unifies quality, durability, and sustainability with a QA/QC decision and
maps to IFC + IDS. Not a new ML algorithm.

## 2. Dataset inventory (summary)

Full details: `outputs/dataset_inventory.md` / `.csv`.

| ID | Dataset | Role | Public source |
|---|---|---|---|
| D1 | UCI Concrete Compressive Strength (Yeh 1998), 1030×9 | **Strength model (primary)** | archive.ics.uci.edu/dataset/165 |
| D2 | Compiled Concrete EPD dataset, U.S.A. (Broyles et al. 2024) | **Carbon benchmarks** | data.mendeley.com/datasets/r4jgxk2mhn |
| D3 | SCM-RAC multi-output concrete | Optional durability | github.com/prayogohandy/SCM-RAC-Concrete-Prediction |
| D4 | ICE database (Hammond & Jones 2008) | Carbon factors | doi.org/10.1680/ener.2008.161.2.87 |
| D5 | NRMCA Industry-Average EPD / GSA-FHWA limits | Carbon-class anchors | nrmca.org |
| D6 | buildingSMART Sample-Test-Files | Optional IFC mapping demo | github.com/buildingSMART/Sample-Test-Files |

## 3. Availability checks (run live)

These cells confirm that the public sources are reachable and load the primary dataset.
All include error-handling fallbacks so the notebook never hard-fails in Colab.

In [ ]:
import requests

SOURCES = {
    'UCI .xls': 'https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls',
    'Mendeley EPD api': 'https://data.mendeley.com/public-api/datasets/r4jgxk2mhn/files?folder_id=root&version=3',
    'SCM-RAC repo': 'https://github.com/prayogohandy/SCM-RAC-Concrete-Prediction',
    'bSI Sample-Test-Files': 'https://github.com/buildingSMART/Sample-Test-Files',
}
for name, url in SOURCES.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=30)
        code = r.status_code
        if code >= 400:  # some hosts dislike HEAD; retry GET headers only
            r = requests.get(url, stream=True, timeout=30); code = r.status_code
        print(f'{name:24s} -> HTTP {code}')
    except Exception as e:
        print(f'{name:24s} -> ERROR {e}')

### 3a. Load the primary strength dataset (D1)
Uses the project module if available, else downloads directly, else falls back to OpenML.

In [ ]:
import pandas as pd
df = None
# Preferred: project pipeline.
try:
    sys.path.insert(0, os.getcwd())
    from src import data_download, data_cleaning
    raw = data_download.download_concrete_strength()
    clean = data_cleaning.clean(raw)
    df = pd.read_csv(clean)
    print('Loaded via project pipeline:', df.shape)
except Exception as e:
    print('Project pipeline unavailable, trying direct/OpenML fallback:', e)
    try:
        url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls'
        df = pd.read_excel(url)
    except Exception as e2:
        print('Direct .xls failed, using OpenML:', e2)
        from sklearn.datasets import fetch_openml
        df = fetch_openml(name='Concrete_Data', version=1, as_frame=True).frame
df.head()

In [ ]:
# Quick sanity summary of the strength dataset.
print('Shape:', df.shape)
display(df.describe().T[['mean','std','min','max']]) if 'display' in dir() else print(df.describe().T)

### 3b. Inspect the EPD dataset header (D2) without a full download
The compiled EPD CSV is large (~81 MB). We fetch only the first bytes to confirm the schema.

In [ ]:
import requests, io
try:
    api = 'https://data.mendeley.com/public-api/datasets/r4jgxk2mhn/files?folder_id=root&version=3'
    files = requests.get(api, timeout=30).json()
    csv_file = next(f for f in files if f['filename'].lower().endswith('.csv'))
    dl = csv_file['content_details']['download_url']
    # Range request for just the header + a few rows.
    hdr = requests.get(dl, headers={'Range': 'bytes=0-4000'}, timeout=60)
    first_line = hdr.text.splitlines()[0]
    cols = [c.strip() for c in first_line.split(',')]
    print(f'EPD file: {csv_file["filename"]} (~{csv_file["size"]/1e6:.0f} MB)')
    print('First columns:', cols[:24])
except Exception as e:
    print('EPD header check failed (will be handled in notebook 03):', e)

## 4. What this notebook produced

- Confirmed reachability of all public sources (UCI, Mendeley EPD, SCM-RAC, buildingSMART).
- Loaded the **primary strength dataset** (1030×9) via the project pipeline (with fallbacks).
- Verified the **EPD dataset schema** (compressive strength psi, curation time, declared unit,
  product components, A1–A3 GWP, plant region) without a full download.
- Persistent inventory artefacts already in `outputs/`: `literature_matrix.csv`,
  `literature_summary.md`, `research_gap_and_contributions.md`, `dataset_inventory.md`, `dataset_inventory.csv`.

Next: **notebook 02** trains the strength model and generates passports; **notebook 03** does carbon
classification and the OpenBIM/IFC + IDS mapping.